In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
# 3. Install kohya_ss sd-scripts
import os

%cd /content
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!git checkout sdxl

In [ ]:
# 4. Install dependencies
%cd /content/sd-scripts

# Install compatible versions
!pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu118
!pip install "numpy<2"
!pip install --force-reinstall opencv-python==4.8.1.78
!pip install -r requirements.txt
!pip install xformers --index-url https://download.pytorch.org/whl/cu118

print("\n✓ Dependencies installed")
!python -c "import torch; print(f'torch: {torch.__version__}')"
!python -c "import cv2; print(f'cv2: {cv2.__version__}')"

In [ ]:
# 5. Download SDXL base model
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="stabilityai/stable-diffusion-xl-base-1.0",
    local_dir="/content/sdxl-base",
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# 6. Configure paths and verify dataset
import os
from pathlib import Path

# Hardcoded paths for your setup
DRIVE_BASE = "/content/drive/MyDrive"
TRAIN_DATA_DIR = f"{DRIVE_BASE}/LoRA Photos/train"
OUTPUT_DIR = f"{DRIVE_BASE}/LoRA Photos/output"
MODEL_PATH = "/content/sdxl-base"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths
print("Checking paths...")
print(f"Train data: {TRAIN_DATA_DIR}")
print(f"  - wesley/: {Path(TRAIN_DATA_DIR, 'wesley').exists()}")
print(f"  - person/: {Path(TRAIN_DATA_DIR, 'person').exists()}")
print(f"Output: {OUTPUT_DIR}")

# Count images
wesley_dir = Path(TRAIN_DATA_DIR, 'wesley')
person_dir = Path(TRAIN_DATA_DIR, 'person')
wesley_count = len(list(wesley_dir.glob('*.jpg'))) + len(list(wesley_dir.glob('*.png')))
person_count = len(list(person_dir.glob('*.jpg'))) + len(list(person_dir.glob('*.png')))
print(f"\nDataset: {wesley_count} subject images, {person_count} regularization images")

In [ ]:
# 7. Create dataset TOML config
dataset_config = f"""
[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

[[datasets]]
resolution = 1024
batch_size = 4
enable_bucket = true
bucket_no_upscale = true

  [[datasets.subsets]]
  image_dir = "{TRAIN_DATA_DIR}/wesley"
  num_repeats = 5
  class_tokens = "wesley_kamau"

  [[datasets.subsets]]
  image_dir = "{TRAIN_DATA_DIR}/person"
  num_repeats = 1
  is_reg = true
  class_tokens = "person"
"""

config_path = "/content/dataset_config.toml"
with open(config_path, "w") as f:
    f.write(dataset_config)

print(f"Dataset config saved to {config_path}")
print(dataset_config)

In [ ]:
# 8. Create caption files for regularization images
from pathlib import Path

reg_dir = Path(TRAIN_DATA_DIR) / "person"
reg_images = list(reg_dir.glob("*.png")) + list(reg_dir.glob("*.jpg"))

created = 0
for img_path in reg_images:
    txt_path = img_path.with_suffix(".txt")
    if not txt_path.exists():
        txt_path.write_text("person")
        created += 1

print(f"Created {created} caption files for regularization images")
print(f"Total regularization images: {len(reg_images)}")

In [ ]:
# 9. Build and display training command
%cd /content/sd-scripts

train_cmd = f"""
accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \\
  --pretrained_model_name_or_path="{MODEL_PATH}" \\
  --dataset_config="{config_path}" \\
  --output_dir="{OUTPUT_DIR}" \\
  --output_name="wesley_kamau_lora" \\
  --network_module="networks.lora" \\
  --network_dim=32 \\
  --network_alpha=16 \\
  --learning_rate=1e-4 \\
  --lr_scheduler="cosine" \\
  --lr_warmup_steps=100 \\
  --max_train_steps=3200 \\
  --optimizer_type="AdamW8bit" \\
  --mixed_precision="fp16" \\
  --save_every_n_steps=500 \\
  --save_model_as="safetensors" \\
  --min_snr_gamma=5.0 \\
  --noise_offset=0.05 \\
  --cache_latents \\
  --cache_latents_to_disk \\
  --gradient_checkpointing \\
  --xformers \\
  --seed=42
"""

print("Training command:")
print(train_cmd)

In [ ]:
# 10. START TRAINING (6-8 hours for 3200 steps)
!{train_cmd}

In [ ]:
# 11. Find latest checkpoint
from pathlib import Path

checkpoints = sorted(Path(OUTPUT_DIR).glob("*.safetensors"))
if checkpoints:
    latest_lora = str(checkpoints[-1])
    print(f"Using LoRA: {latest_lora}")
else:
    print("No checkpoints found yet!")
    latest_lora = None

In [ ]:
# 12. Test the trained LoRA
if latest_lora:
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
    import torch
    import os

    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    ).to("cuda")
    
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    
    pipe.load_lora_weights(
        os.path.dirname(latest_lora),
        weight_name=os.path.basename(latest_lora)
    )
    
    test_prompts = [
        "wesley_kamau, person, male, professional headshot, soft lighting, instagram profile photo style",
        "wesley_kamau, person, male, casual portrait, natural lighting, warm smile",
        "wesley_kamau, person, male, studio portrait, dramatic lighting, looking at camera"
    ]
    
    for i, prompt in enumerate(test_prompts):
        image = pipe(
            prompt=prompt,
            negative_prompt="blurry, distorted, deformed, bad anatomy, ugly",
            num_inference_steps=30,
            guidance_scale=7.5,
            width=1024,
            height=1024,
        ).images[0]
        
        output_path = f"{OUTPUT_DIR}/test_{i+1}.png"
        image.save(output_path)
        print(f"Saved: {output_path}")
        display(image)

In [ ]:
# 13. List output files
!ls -la "{OUTPUT_DIR}"